# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
import optuna
import trackio
import matplotlib.pyplot as plt
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [2]:
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

train_df = pd.read_csv('Dataset/train.csv')
test_df = pd.read_csv('Dataset/test.csv')

categorical_cols = ['city', 'location', 'Status', 'property_type']
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]])
    le.fit(combined)
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

with open('models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

X_train = train_df.drop('price', axis=1)
y_train = train_df['price']
X_test = test_df.drop('price', axis=1)
y_test = test_df['price']

## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [3]:
run = trackio.init(project="UrbanNest-PropTech-Optimization")

param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 8]
}
grid_history = []
rand_history = []
optuna_history = []

rf = RandomForestRegressor(random_state=42)
#1. Grid Search 
start_time = time.time()
gs = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
gs.fit(X_train, y_train)
gs_time = time.time() - start_time

results = gs.cv_results_
best_err = float('inf')
for err in -results['mean_test_score']:
    if err < best_err: best_err = err
    grid_history.append(best_err)

print(f"Grid Search Best MAE: {-gs.best_score_:.2f}")
print(f"Grid Search Best Params: {gs.best_params_}")
trackio.log({
    'method': 'GridSearch',
    'iterations': len(results['params']),
    'time_taken': gs_time,
    'best_cv_mae': float(-gs.best_score_),
    'best_params': str(gs.best_params_)
})

#2. Random Search 
start_time = time.time()
from scipy.stats import randint
rs = RandomizedSearchCV(rf, param_distributions={'n_estimators': randint(50, 201), 'max_depth': randint(10, 31), 'min_samples_split': randint(2, 11)}, n_iter=60, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42)
rs.fit(X_train, y_train)
rs_time = time.time() - start_time

results_rs = rs.cv_results_
best_err = float('inf')
for err in -results_rs['mean_test_score']:
    if err < best_err: best_err = err
    rand_history.append(best_err)
    
print(f"Random Search Best MAE: {-rs.best_score_:.2f}")
print(f"Random Search Best Params: {rs.best_params_}")
trackio.log({
    'method': 'RandomSearch',
    'iterations': len(results_rs['params']),
    'time_taken': rs_time,
    'best_cv_mae': float(-rs.best_score_),
    'best_params': str(rs.best_params_)
})

#3. Bayesian Optimization (Optuna)
optuna_start_time = time.time()
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 10, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    mae = -score.mean()
    
    if not hasattr(objective, 'best_err'): objective.best_err = float('inf')
    if mae < objective.best_err: objective.best_err = mae
    optuna_history.append(objective.best_err)
    return mae

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=60)
optuna_time = time.time() - optuna_start_time

print(f"Optuna Best MAE: {study.best_value:.2f}")
print(f"Optuna Best Params: {study.best_params}")
trackio.log({
    'method': 'Optuna',
    'iterations': len(study.trials),
    'time_taken': optuna_time,
    'best_cv_mae': float(study.best_value),
    'best_params': str(study.best_params)
})

* Trackio project initialized: UrbanNest-PropTech-Optimization
* Trackio metrics logged to: C:\Users\DELL\.cache\huggingface\trackio
* Created new run: eager-mountain-3


Grid Search Best MAE: 13270.76
Grid Search Best Params: {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}


[I 2026-04-15 00:12:02,537] A new study created in memory with name: no-name-4c74ee56-0f43-4ef4-99e5-da2c69cd95e4


Random Search Best MAE: 13309.78
Random Search Best Params: {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}


[I 2026-04-15 00:12:06,730] Trial 0 finished with value: 13535.497553960819 and parameters: {'n_estimators': 198, 'max_depth': 16, 'min_samples_split': 2}. Best is trial 0 with value: 13535.497553960819.


...

[I 2026-04-15 00:17:38,089] Trial 56 finished with value: 13727.348763342638 and parameters: {'n_estimators': 175, 'max_depth': 26, 'min_samples_split': 7}. Best is trial 55 with value: 13268.626195376715.


[I 2026-04-15 00:17:42,908] Trial 57 finished with value: 13283.799137776883 and parameters: {'n_estimators': 193, 'max_depth': 27, 'min_samples_split': 2}. Best is trial 55 with value: 13268.626195376715.


[I 2026-04-15 00:17:47,389] Trial 58 finished with value: 13454.389276285636 and parameters: {'n_estimators': 182, 'max_depth': 28, 'min_samples_split': 4}. Best is trial 55 with value: 13268.626195376715.


[I 2026-04-15 00:17:50,615] Trial 59 finished with value: 13832.856092996231 and parameters: {'n_estimators': 129, 'max_depth': 24, 'min_samples_split': 8}. Best is trial 55 with value: 13268.626195376715.


Optuna Best MAE: 13268.63
Optuna Best Params: {'n_estimators': 190, 'max_depth': 26, 'min_samples_split': 2}


## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [4]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(grid_history)+1), grid_history, label='Grid Search')
plt.plot(range(1, len(rand_history)+1), rand_history, label='Random Search')
plt.plot(range(1, len(optuna_history)+1), optuna_history, label='Optuna')
plt.xlabel('Number of Iterations / Trials')
plt.ylabel('Best Mean CV MAE')
plt.title('Compute Budget vs Error')
plt.legend()
plt.savefig('plots/trials_vs_error.png')
plt.show()

fig = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.savefig('plots/optuna_hyperparameter_space.png')
plt.show()

C:\Users\DELL\AppData\Local\Temp\ipykernel_14644\230524325.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\DELL\AppData\Local\Temp\ipykernel_14644\230524325.py:12: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  fig = optuna.visualization.matplotlib.plot_optimization_history(study)


C:\Users\DELL\AppData\Local\Temp\ipykernel_14644\230524325.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [5]:
best_mae = min(-gs.best_score_, -rs.best_score_, study.best_value)
if best_mae == -gs.best_score_: best_params = gs.best_params_
elif best_mae == -rs.best_score_: best_params = rs.best_params_
else: best_params = study.best_params

print("Best Parameters Overall:", best_params)
final_model = RandomForestRegressor(**best_params, random_state=42)
final_model.fit(X_train, y_train)

test_preds = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_preds)
print(f"Final Test MAE: {test_mae:.2f}")

with open('models/best_rf_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
print("Model saved to models/best_rf_model.pkl")

Best Parameters Overall: {'n_estimators': 190, 'max_depth': 26, 'min_samples_split': 2}


Final Test MAE: 12419.50
Model saved to models/best_rf_model.pkl
